# Batch Deployment of ML Models

In this notebook we will prepare our code for batch deployment. We will use the MLFlow server we set up in the previous notebook to store our models and the predictions. Afterwards we will refactor our code into python scripts and schedule the execution of the predictions with prefect.

Batch predictions are predictions that are executed on a regular basis. For example, you might want to predict the sales for the next day every night at 12am. Batch predictions are often used in analytics when there is no need for real-time predictions. 

In this example we want to analyse if there are differences between the actual duration of a trip and the duration that was predicted by the model. It is not the best example for batch predictions, but it is good enough to show how it works.

First let's import the libraries we will need:

In [6]:
import os
import uuid
import pandas as pd

import mlflow

from dotenv import load_dotenv
import os

load_dotenv()

MLFLOW_TRACKING_URI=os.getenv("MLFLOW_TRACKING_URI")
RUN_ID=os.getenv("RUN_ID")
BUCKET_NAME=os.getenv("BUCKET_NAME")
SA_KEY=os.getenv("GOOGLE_SA_KEY")

In [ ]:
filename = "data/green_tripdata_2021-01.parquet"

As mentioned in the the last notebook the code to run the model with mlflow is shown in the MLFlow UI if you click on the run and of course we have to set the `GOOGLE_APPLICATION_CREDENTIALS` again:

In [9]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SA_KEY

## Prepare the code for batch deployment

Normally data comes with unique identifiers. In our case we don't have any unique identifiers so we will create one for each row. We will use the `uuid` library to them.

In [ ]:
def generate_uuids(n):
    ride_ids = []
    for i in range(n):
        ride_ids.append(str(uuid.uuid4()))
    return ride_ids


Let's write a function that loads the data, creates the trip_duration column and adds the unique identifier:

In [ ]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['trip_duration_minutes'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.trip_duration_minutes = df.trip_duration_minutes.dt.total_seconds() / 60
    df = df[(df.trip_duration_minutes >= 1) & (df.trip_duration_minutes <= 60)]
    
    df['ride_id'] = generate_uuids(len(df))

    return df

We need to prepare the data the same way as in the notebook before. We will use the same function to do that:

In [ ]:
def preprocess(df):
    df = df.copy()
    categorical_features = ["PULocationID", "DOLocationID"]
    df[categorical_features] = df[categorical_features].astype(str)
    
    df['trip_route'] = df["PULocationID"] + "_" + df["DOLocationID"]
    dicts = df[['trip_route', 'trip_distance']].to_dict(orient='records')
    
    return dicts

And now we can write the function that loads the model, makes the predictions and creates a new dataframe with the predictions:

In [ ]:
def load_model(run_id):
    logged_model = f'runs:/{run_id}/model'
    # Load model as a PyFuncModel.
    loaded_model = mlflow.pyfunc.load_model(logged_model)
    return loaded_model


def save_results(df, y_pred, run_id, output_filename):
    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['lpep_pickup_datetime'] = df['lpep_pickup_datetime']
    df_result['PULocationID'] = df['PULocationID']
    df_result['DOLocationID'] = df['DOLocationID']
    df_result['actual_duration'] = df['trip_duration_minutes']
    df_result['predicted_duration'] = y_pred
    df_result['diff'] = df_result['actual_duration'] - df_result['predicted_duration']
    df_result['model_version'] = run_id
    df_result.to_parquet(output_filename, index=False)



def apply_model(filename, run_id, output_filename):
    df = read_dataframe(filename)
    dicts = preprocess(df)
    
    loaded_model = load_model(run_id)
    y_pred = loaded_model.predict(dicts)
    
    save_results(df, y_pred, run_id, output_filename)


In [ ]:
apply_model(filename, RUN_ID, "data/predictions.parquet")

Now we can refactor this code into python scripts. In the folder called `src/batch/` we will create a file named `predict.py`. We will copy the code from the functions above into the scripts. What we will add in the end is a `def run()` function that will run the whole script. The `run()` function will be called in the `if __name__ == "__main__":` block. And will be parameterized with click:

```python
@click.command()
@click.option("--filename", help="Path to the input parquet file")
@click.option("--run_id", help="MLflow run ID")
@click.option("--output_filename", help="Path to the output parquet file")
@click.option("--google_sa_key", help="Path to the Google SA key")
def run(filename, run_id, output_filename, google_sa_key):
    filename = filename
    output_filename = output_filename
    run_id = run_id
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = google_sa_key
    apply_model(filename,
                run_id,
                output_filename)
    
    

if __name__ == "__main__":
    run()
```

Now you can run the script with the following command:

```bash
python src/batch/predict.py --help
```

Also feel free to change the filename into a command that downloads also other data from the green taxi dataset. In the end you could also create a Docker image from this script and run it in a container.

## Schedule the predictions with Prefect

But we will schedule the predictions with Prefect. Prefect is a workflow management system that allows you to schedule and orchestrate your workflows. You can read more about Prefect [here](https://docs.prefect.io/core/). And in the repositroy from last week about Workflow Orchestration.

You can start the Prefect server with the following command:

```bash
prefect orion start
```

Before we start create a new GCS Bucket and two folders in it. One for the input data and one for the output data. Pandas allows us to read and write parquet files directly from and to GCS. So we will use this feature to read the data and write the predictions to GCS. 

First we will create a new python file called `predict_prefect.py` and copy the code from the `predict.py` script into it. But we will remove the click options and add the parameters as normal python variables. 

Instead we will import prefect `flow` and `task` and turn the `run()` function into a `flow`:

```python 
@flow(name="Predict Green Taxi Trip Duration")
def run(bucket_name:str, 
        run_id:str, 
        google_sa_key:str,
        MLFLOW_TRACKING_URI,
        run_date:datetime = None):
```

We will add `logger` for more output in the prefect logs.

```python
    logger = get_run_logger()
    logger.info(f"Running with parameters: bucket_name={bucket_name}, run_id={run_id}, google_sa_key={google_sa_key}, run_date={run_date}")
    path = os.path.dirname(__file__)
    logger.info(f"Current path: {path}")
```

To transform it into a scheduled flow we will add a schedule we will get the month from the schedule with the `get_run_context()` function from `prefect`. With this we will always get the current month - 1 as the month for the data we want to download. But with the `run_date`  as parameter we could also specify a date. This is useful if we want to rerun the flow for a specific month. 




```python
    if run_date is None:
        ctx = get_run_context()
        date = ctx.flow_run.expected_start_time
        month = date.month - 1
        
    year = 2021

```

The next part is the connection to the `GCS Bucket`, the upload of the data and the prediction:

```python
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = google_sa_key
    df_input = pd.read_parquet(f"https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{year}-{month:02d}.parquet")
    df_input.to_parquet(f"gs://{bucket_name}/raw/green_tripdata_{year}-{month:02d}.parquet")
    
    filename = f"gs://{bucket_name}/raw/green_tripdata_{year}-{month:02d}.parquet"
    output_filename = f"gs://{bucket_name}/predictions/green_tripdata_{year}-{month:02d}.parquet"
    run_id = run_id
    apply_model(filename,
                run_id,
                output_filename,
                MLFLOW_TRACKING_URI)
```

Last time we directly created the yml file for Prefect. This time we will use a python file `prefect_deploy.py` for creating a Deployment:

```python
from prefect.deployments import Deployment
from prefect.server.schemas.schedules import CronSchedule
from predict_prefect import run
from dotenv import load_dotenv
import os

load_dotenv()

MLFLOW_TRACKING_URI=os.getenv("MLFLOW_TRACKING_URI")
RUN_ID=os.getenv("RUN_ID")
BUCKET_NAME=os.getenv("BUCKET_NAME")
GOOGLE_SA_KEY=os.getenv("GOOGLE_SA_KEY")


deployment = Deployment.build_from_flow(
    flow=run,
    name="ride_duration_prediction",
    parameters={
        "bucket_name": BUCKET_NAME,
        "run_id": RUN_ID,
        "google_sa_key": GOOGLE_SA_KEY,
        "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI
    },
    schedule=CronSchedule(cron="0 3 2 * *"),
    tags=["batch", "predict", "prefect"]
)

deployment.apply()
```
You can add the parameters as environment variables or directly in the file.
And than run that python file:

```bash
python src/batch/prefect_deploy.py
```

Don't forget to start the prefect agent:

```bash
 prefect agent start -q 'default'
```

Feel freee to start a `Quick Run` in the [Prefect UI](http://127.0.0.1:4200/). And check the logs. 